# MOSS-TTS v1.5 — Kaggle T4×2 Production Demo / Demo Production Kaggle T4×2

Run the original BF16 safetensors on **GPU T4 x2** with no GGUF, no quantization and no model modification. / Chạy safetensors BF16 nguyên bản trên **GPU T4 x2**, không GGUF, không quantization, không sửa model.

> Independent engineering qualification project, not an official OpenMOSS release. / Dự án kỹ thuật độc lập, không phải bản phát hành chính thức của OpenMOSS.
>
> Publication is **not** complete. Publication is gated on this notebook plus a separate human listening review. / Chưa phát hành. Việc phát hành còn phụ thuộc kết quả chạy notebook này và đợt nghe thủ công.

## Before you run / Trước khi chạy

1. Import this notebook from GitHub repository `dangkhoa2016/MOSS-TTS-v1.5-On-Kaggle-T4x2-GPU`.
2. Select **Accelerator → GPU T4 x2**.
3. Attach Kaggle Model `dangkhoa2016/openmoss-team-moss-tts-v1-5`.
4. Keep Internet ON for the source checkout and dependency bootstrap.
5. Set the Kaggle environment variable **`MOSS_TTS_SOURCE_REF`** to an **exact 40-character commit SHA** (release-candidate qualification) or to a release tag such as `v1.0.0` after tagging. Mutable refs such as `main` are rejected, so the same notebook always runs the same source. / Đặt biến môi trường `MOSS_TTS_SOURCE_REF` bằng SHA 40 ký tự hoặc thẻ phát hành; `main` bị từ chối.
6. Run `Restart Session → Run All`. The bootstrap may upgrade installed packages, but every runtime step below runs in a fresh subprocess, so the documented flow needs no extra restart. / Chạy `Restart Session → Run All`.

## Step 1 — Immutable public source pin / Ghim mã nguồn bất biến

Expected markers: `PUBLIC_SOURCE_HEAD=<40-char-sha>`, `PUBLIC_SOURCE_PIN=PASS`. A mutable ref such as `main` fails here by design.

In [ ]:
from pathlib import Path
import json, os, re, shutil, subprocess, sys

REPO_URL = "https://github.com/dangkhoa2016/MOSS-TTS-v1.5-On-Kaggle-T4x2-GPU.git"
ROOT = Path("/kaggle/working/moss-tts-v15-public")
REPORTS = ROOT / "results" / "notebook"
SOURCE_REF = os.environ.get("MOSS_TTS_SOURCE_REF", "").strip()
COMMIT_SHA = re.compile(r"^[0-9a-f]{40}$")
RELEASE_TAG = re.compile(r"^v\d+\.\d+\.\d+(?:-[0-9A-Za-z.]+)?$")

if not (COMMIT_SHA.match(SOURCE_REF) or RELEASE_TAG.match(SOURCE_REF)):
    raise SystemExit(
        "MOSS_TTS_SOURCE_REF must be an exact 40-character commit SHA or a release tag such as v1.0.0; "
        "mutable refs such as 'main' are not accepted as release authority. Got: " + repr(SOURCE_REF)
    )

if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(ROOT)], check=True)
subprocess.run(["git", "fetch", "--depth", "1", "origin", SOURCE_REF], cwd=ROOT, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ROOT, check=True)

RESOLVED_HEAD = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if COMMIT_SHA.match(SOURCE_REF) and RESOLVED_HEAD != SOURCE_REF:
    raise SystemExit(f"source pin mismatch: requested {SOURCE_REF}, checked out {RESOLVED_HEAD}")

SOURCE_PINNED = RESOLVED_HEAD == SOURCE_REF or bool(RELEASE_TAG.match(SOURCE_REF))
assert SOURCE_PINNED, f"source pin could not be proven for {SOURCE_REF} / {RESOLVED_HEAD}"
print(f"PUBLIC_SOURCE_REF={SOURCE_REF}")
print(f"PUBLIC_SOURCE_HEAD={RESOLVED_HEAD}")
print("PUBLIC_SOURCE_PIN=PASS")

## Step 2 — Dependency baseline bootstrap / Cài đặt baseline phụ thuộc

`requirements/kaggle.txt` is installed from the pinned source revision and every declared range is verified against the interpreter that later runs the qualification scripts.

Expected markers: `DEPENDENCY_BASELINE=PASS`, `KAGGLE_KERNEL_RESTART_REQUIRED=NO`.

In [ ]:
from importlib import metadata as importlib_metadata
from packaging.requirements import Requirement

REQUIREMENTS = ROOT / "requirements" / "kaggle.txt"
declared = [
    Requirement(line)
    for line in REQUIREMENTS.read_text().splitlines()
    if line.strip() and not line.strip().startswith(("#", "-"))
]
installed_before = {req.name: importlib_metadata.version(req.name) for req in declared}

subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS)], check=True)

installed_after = {req.name: importlib_metadata.version(req.name) for req in declared}
violations = [
    f"{req.name}{req.specifier} != {installed_after[req.name]}"
    for req in declared
    if not req.specifier.contains(installed_after[req.name], prereleases=True)
]
changed = sorted(name for name in installed_after if installed_before.get(name) != installed_after[name])

for req in declared:
    print(f"DEPENDENCY {req.name}{req.specifier} installed={installed_after[req.name]}")
assert not violations, "dependency baseline violated: " + ", ".join(violations)
print("DEPENDENCY_BASELINE=PASS")
# Every runtime step below runs in a fresh subprocess started after this install, so the
# upgraded baseline is what actually executes; no extra kernel restart is required.
print("KAGGLE_KERNEL_RESTART_REQUIRED=NO")
if changed:
    print("KAGGLE_KERNEL_RESTART_ADVISED_PACKAGES=" + ",".join(changed))

## Step 3 — GPU T4×2 preflight / Kiểm tra GPU

Expected marker: `GPU_T4X2=PASS`. The runner accepts exactly two devices whose normalized name is `tesla t4`.

In [ ]:
REPORTS.mkdir(parents=True, exist_ok=True)

preflight_run = subprocess.run(
    [sys.executable, "scripts/preflight.py", "--report", str(REPORTS / "preflight.json")],
    cwd=ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(preflight_run.stdout.strip())
preflight_report = json.loads((REPORTS / "preflight.json").read_text())
assert preflight_report["status"] == "PASS"
assert preflight_report["gpu_count"] == 2
assert preflight_report["normalized_gpu_names"] == ["tesla t4", "tesla t4"]
assert "GPU_T4X2=PASS" in preflight_run.stdout
print("PREFLIGHT_REPORT=" + json.dumps(preflight_report, sort_keys=True))
print("GPU_T4X2=PASS")

## Step 4 — Checkpoint identity, dtype and format evidence / Bằng chứng danh tính checkpoint

Expected markers: `MODEL_IDENTITY_AND_SAFETENSORS=PASS`, `QUANTIZATION=NONE`, `GGUF=NO`.

The marker is deliberately named for identity plus format evidence. Cryptographic proof that the attached files are the unmodified upstream originals additionally requires a confirmed `references/checkpoint-sha256.json`; the report states which of `PASS` or `UNVERIFIED` applies.

In [ ]:
inventory_run = subprocess.run(
    [
        sys.executable,
        "scripts/inventory_model.py",
        "--root",
        "/kaggle/input/models",
        "--report",
        str(REPORTS / "inventory.json"),
    ],
    cwd=ROOT,
    check=True,
    capture_output=True,
    text=True,
)
inventory_report = json.loads((REPORTS / "inventory.json").read_text())

assert inventory_report["status"] == "PASS"
assert inventory_report["identity"]["declared_dtype"] == "bfloat16"
assert inventory_report["checkpoint_dtype_verification"]["status"] == "PASS"
assert inventory_report["checkpoint_dtype_verification"]["weights_materialized"] is False
assert inventory_report["format"]["gguf"] is False
assert inventory_report["quantization"] == "none"
assert inventory_report["checkpoint_sha256_reference"]["status"] in {"PASS", "UNVERIFIED"}

print("SHA256_REFERENCE=" + inventory_report["checkpoint_sha256_reference"]["status"])
print("CHECKPOINT_SHARDS=" + str(inventory_report["shard_count"]))
print("QUANTIZATION=" + inventory_report["quantization"].upper())
print("GGUF=NO")
print("MODEL_IDENTITY_AND_SAFETENSORS=PASS")

## Step 5 — Reviewer-facing synthesis / Âm thanh cho reviewer

The canonical runner verifies checkpoint identity **before** loading weights, then compares the loaded module set against the safetensors weight map. English, Vietnamese and Vietnamese-English **code-switch** samples are rendered inline.

In [ ]:
from IPython.display import Audio, display

CASES = [
    ("en", "English", "Today we are testing clear speech on two Tesla T4 GPUs."),
    ("vi", "Vietnamese", "Xin chào. Đây là bài kiểm tra tổng hợp giọng nói tiếng Việt trên hai GPU Tesla T4."),
    (
        "codeswitch",
        "Vietnamese",
        "Xin chào, hôm nay mình đang kiểm tra Moss TTS trên Kaggle, and this sentence switches between Vietnamese and English.",
    ),
]

generation_reports = {}
for case_id, language, text in CASES:
    wav = REPORTS / f"{case_id}.wav"
    report_path = REPORTS / f"{case_id}.json"
    completed = subprocess.run(
        [
            sys.executable,
            "scripts/generate.py",
            "--model-root",
            "/kaggle/input/models",
            "--text",
            text,
            "--language",
            language,
            "--output",
            str(wav),
            "--report",
            str(report_path),
        ],
        cwd=ROOT,
        check=True,
        capture_output=True,
        text=True,
    )
    generation_reports[case_id] = json.loads(report_path.read_text())
    print(f"{case_id}: audio_bytes={wav.stat().st_size} stdout_tail={completed.stdout.strip().splitlines()[-1]}")
    display(Audio(str(wav)))

## Step 6 — IPA pronunciation control / Điều khiển phát âm IPA

Raw acronyms may be spelled out by the model. This sample uses the upstream-supported **IPA** span feature; it is not a model patch and the repository does not silently normalize user text. / Mẫu này dùng feature **IPA** của upstream, không phải model patch.

In [ ]:
ipa_text = "Moss TTS is running with /biː ɛf sɪkˈstiːn/ precision on two Tesla T4 GPUs."
ipa_wav = REPORTS / "ipa.wav"
ipa_report_path = REPORTS / "ipa.json"
subprocess.run(
    [
        sys.executable,
        "scripts/generate.py",
        "--model-root",
        "/kaggle/input/models",
        "--text",
        ipa_text,
        "--language",
        "English",
        "--output",
        str(ipa_wav),
        "--report",
        str(ipa_report_path),
    ],
    cwd=ROOT,
    check=True,
    capture_output=True,
    text=True,
)
generation_reports["ipa"] = json.loads(ipa_report_path.read_text())
print(f"ipa: audio_bytes={ipa_wav.stat().st_size}")
display(Audio(str(ipa_wav)))

## Step 7 — Derived scorecard and final machine-readable qualification / Bảng điểm suy ra từ báo cáo

Every field below is derived from the machine-readable reports written by the previous steps. No qualification boolean is declared by hand, and `KAGGLE_PRODUCTION_DEMO=PASS` is printed only after every technical gate is asserted. Human listening review stays a separate gate.

In [ ]:
all_reports = list(generation_reports.values())

scorecard = {
    "public_source_ref": SOURCE_REF,
    "public_source_head": RESOLVED_HEAD,
    "public_source_pin": RESOLVED_HEAD == SOURCE_REF or bool(RELEASE_TAG.match(SOURCE_REF)),
    "gpu_t4x2": preflight_report["status"] == "PASS" and preflight_report["gpu_count"] == 2,
    "model_identity_and_safetensors": inventory_report["status"] == "PASS",
    "declared_dtype_bf16": inventory_report["identity"]["declared_dtype"] == "bfloat16",
    "checkpoint_dtype_verification": inventory_report["checkpoint_dtype_verification"]["status"] == "PASS",
    "checkpoint_sha256_reference": inventory_report["checkpoint_sha256_reference"]["status"],
    "gguf": inventory_report["format"]["gguf"],
    "quantization": inventory_report["quantization"],
    "model_patch": any(report["weight_map_match"]["status"] != "PASS" for report in all_reports),
    "architecture_modified": any(report["weight_map_match"]["status"] != "PASS" for report in all_reports),
    "audio_cases": sorted(generation_reports),
    "generated_steps": {case: report["generated_steps"] for case, report in generation_reports.items()},
    "generation_rtf": {case: report["generation_rtf"] for case, report in generation_reports.items()},
    "decode_rtf": {case: report["decode_rtf"] for case, report in generation_reports.items()},
    "audio_duration_seconds": {case: report["audio_duration_seconds"] for case, report in generation_reports.items()},
    "gpu_peak_alloc_gib": {
        case: [report["gpu0_peak_alloc_gib"], report["gpu1_peak_alloc_gib"]]
        for case, report in generation_reports.items()
    },
    "runtime_versions": all_reports[0]["provenance"]["package_versions"],
    "checkpoint_hashes": inventory_report["files"],
    "human_listening_review": "PENDING",
}

assert scorecard["public_source_pin"] is True
assert scorecard["gpu_t4x2"] is True
assert scorecard["model_identity_and_safetensors"] is True
assert scorecard["declared_dtype_bf16"] is True
assert scorecard["checkpoint_dtype_verification"] is True
assert scorecard["checkpoint_sha256_reference"] in {"PASS", "UNVERIFIED"}
assert scorecard["gguf"] is False
assert scorecard["quantization"] == "none"
assert scorecard["model_patch"] is False
assert scorecard["architecture_modified"] is False
assert set(scorecard["audio_cases"]) == {"en", "vi", "codeswitch", "ipa"}
assert all(report["signal"]["status"] == "PASS" for report in all_reports)
assert all(report["model_identity"]["status"] == "PASS" for report in all_reports)

print(json.dumps(scorecard, indent=2, ensure_ascii=False))
print("QUANTIZATION=" + scorecard["quantization"].upper())
print("GGUF=NO")
print("MODEL_PATCH=NONE")
print("HUMAN_LISTENING_REVIEW=PENDING")
print("KAGGLE_PRODUCTION_DEMO=PASS")